# Defter 23A — Reviewer analyses without retraining
**Theory-guided AI reveals contrasting effects of a century of climate change on tree growth** · Prof. Dr. İlker Ercanlı

**Ne yapar (TR):** Yeni eğitim yok. Defter 20'nin 20 üyeli nihai modeli ve Defter 21'in kaydettiği saha-yıl atıf dosyaları
(`SONUC_21/atif/uyeXX.npz`) ile hakem değerlendirmesindeki sorulara cevap verir:

| Bölüm | Soru | Çıktı (`SONUC_23A/`) |
|---|---|---|
| A1 | Ağ ve halka tahmini **aynı sahalarda, aynı istatistikle, aynı 10° blok bootstrap ile** nasıl karşılaştırılır? | `A1_ayni_saha.csv` |
| A2 | Ayrımın ortaya çıkışı, örneklem değişmeden (dengeli panel ve zincirleme eşleştirme) ve önceden tanımlı ölçütle | `A2_denge_onyil.csv`, `A2_ortaya_cikis.csv` |
| A3 | Ağ sonuçları için 10° mekânsal blok güven aralıkları (tüm atıf sahaları) | `A3_blok_aralik.csv` |
| A4 | Atıf, seri yaşı ve örnekleme bitiş yılına göre değişiyor mu? (yüzyıllık sapmaya duyarlılık) | `A4_yas_orneklem.csv` |
| A5 | Karşı-olgusal iklimde sınırlamalar kaç ayda devreye girdi, sınırlar gevşetilince sonuç değişiyor mu? | `A5_sinirlama_sayim.csv`, `A5_sinirlama_duyarlilik.csv` |
| A6 | Sıcaklık, nem (yağış + toprak nemi) ve VPD için tam faktöriyel (2³ = 8) ayrıştırma ve Shapley payları | `A6_faktoriyel.csv`, `A6_faktoriyel_alan.csv` |
| A7 | Karşı-olgusal girdiler, sahanın gözlenen iklim aralığının ne kadar dışında? | `A7_dagilim_disi.csv` |

**Önce:** Defter 20 ve 21 bitmiş olmalı; `ANALIZ_SON` klasöründe taslak 10'un saha düzeyi halka tahmini (`N1a_uc_olay_kanali_alan.csv`).
**Çalışma zamanı:** L4 (yoksa T4) GPU, yüksek RAM → "Tümünü çalıştır". A5 ve A6 GPU kullanır (4 üye, 1990–2019 halkaları).
**Bitiş satırı:** `==== DEFTER 23A TAMAM ====`

In [ ]:
# --- Setup ---
import os, json, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:                                   # local test outside Colab
    ANA = os.environ.get('ITRDB_AI', '.')
KAYNAK = os.path.join(ANA, 'SONUC_20')                # trained models of Defter 20
CIKTI = os.path.join(ANA, 'SONUC_23A'); os.makedirs(CIKTI, exist_ok=True)
K21 = os.path.join(ANA, 'SONUC_21')                   # attribution files of Defter 21
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(no GPU: Runtime -> change runtime type -> GPU)')

AYAR = dict(
    J=8, emb=8, takson_dusur=0.10, desc_gurultu=0.2, r0_bas_cm=2.0, sifir_mm=0.005,
    blok_yil=5, n_kat=4, bolme_tohum=2026, ic_blok_oran=0.10, agac_onsel=0.3, hizli_deneme=False,
    model='teori_vpd_opt_re',                        # final model (Defter 20)
    yil_son=2019,                                    # ISIMIP3a obsclim/counterclim end year
    kanal_uyeler=None,                               # None: all 20 members for the channel runs; or e.g. 4
    egri_yil=100, egri_cap_cm=[15, 30],              # growth-curve simulation: years, diameter thresholds
)
AYAR.update(json.loads(os.environ.get('D23_AYAR', '{}')))   # override only for local testing
if AYAR['hizli_deneme']: AYAR['max_epoch'] = min(AYAR['max_epoch'], 2)
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

In [ ]:
# --- Data (identical preparation to Defter 2) ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1

clim = d['clim']; soil = d['soil']; gE = d['gE']; NY = clim.shape[1]
YIL1 = YIL0 + NY - 1
# rings need the previous year's climate, so 1901 rings are dropped (as in Defter 2)
k = (d['ring_year'] > YIL0) & (d['ring_year'] <= YIL1)
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k].astype(np.int64), torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
R_site = ser_site[R_ser]; R_tree = ser_tree[R_ser]
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# monthly climate per site: T, log(1+P), soil moisture M, VPD, growing-season weight gE (+ CWD, SPEI-like, below)
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
# --- additional water-stress variables (computed from the same CRU TS and bucket-model data; no download) ---
# climatic water deficit: CWD = PET - AET, with AET = PET x M / Mmax (supply-limited evapotranspiration of the
# VS-Lite bucket, Mmax = 0.76); in units of 100 mm per month
MMAX = 0.76
pet = np.maximum(np.nan_to_num(clim[..., 2]), 0)
cwd = (pet * (1 - np.clip(soil, 0, MMAX) / MMAX) / 100).reshape(S, NY * 12)
# SPEI-like index: 12-month sum of P - PET, standardized per site and calendar month against 1901-1960
wb = (np.nan_to_num(clim[..., 1]) - pet).reshape(S, NY * 12)
cs = np.cumsum(wb, axis=1); wb12 = np.full_like(cs, np.nan); wb12[:, 11:] = cs[:, 11:] - np.concatenate([np.zeros((S, 1)), cs[:, :-12]], 1)
wb12 = wb12.reshape(S, NY, 12); ref = wb12[:, :60]
z12 = (wb12 - np.nanmean(ref, 1, keepdims=True)) / (np.nanstd(ref, 1, keepdims=True) + 1e-6)
z12 = np.nan_to_num(np.clip(z12, -4, 4)).reshape(S, NY * 12)
ozel = np.concatenate([ozel, cwd[..., None].astype(np.float32), z12[..., None].astype(np.float32)], -1)
print(f'Water deficit: median {np.median(cwd)*100:.0f} mm/month; SPEI-like index range {z12.min():.1f} to {z12.max():.1f}')
del pet, cwd, wb, cs, wb12, ref, z12
n_nan = int(np.isnan(ozel).sum())
if n_nan: print(f'WARNING: {n_nan} missing climate values set to the site-month mean')
if n_nan:
    nk = ozel.shape[-1]
    ort = np.nanmean(ozel.reshape(S, NY, 12, nk), axis=1, keepdims=True)
    o4 = ozel.reshape(S, NY, 12, nk); o4[:] = np.where(np.isnan(o4), ort, o4); ozel = o4.reshape(S, NY * 12, nk)
OZ = t(ozel); del ozel

desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
env = d['env']; ENV = t((env - env.mean(0)) / (env.std(0) + 1e-6), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
IC = t(d['site_ic_dogrulama'], torch.bool)            # 10% of sites reserved for early stopping (fixed in Defter 1)
print(f'{NR:,} rings, {NSER:,} series, {NTREE:,} trees, {S} sites, years {YIL0+1}-{YIL1}')

def pencere(idx):
    """24-month climate window (previous January - current December) for rings idx: [B, 24, 5]"""
    s = R_site[idx]; bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

def girdiler(idx):
    X, s = pencere(idx)
    return dict(X=X, rp=R_rp[idx], tree=R_tree[idx], g=site_genus[s], sp=site_species[s], desc=DESC[s],
                env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s])

In [ ]:
# --- Year-block split: 5-year blocks; within each run of 4 consecutive blocks the folds are permuted,
#     so every fold covers the whole period (early, mid-century and the recent warm decades) ---
yillar = np.arange(YIL0 + 1, YIL1 + 1)
blok = (yillar - yillar[0]) // AYAR['blok_yil']; nb = int(blok.max()) + 1
rng = np.random.default_rng(AYAR['bolme_tohum']); kat_blok = np.empty(nb, int)
for b0 in range(0, nb, AYAR['n_kat']):
    ix = np.arange(b0, min(b0 + AYAR['n_kat'], nb)); kat_blok[ix] = rng.permutation(AYAR['n_kat'])[:len(ix)]
yk = np.full(YIL1 + 2, -1, np.int64); yk[yillar] = kat_blok[blok]
YK = t(yk, torch.long)
tablo = pd.DataFrame(dict(year=yillar, block=blok, fold=kat_blok[blok]))
SITE_KAT = t(d['site_blok_kat'], torch.long)          # 10-degree spatial blocks in 5 groups (fixed in Defter 1)
R_kat = YK[R_yil]
for kk in range(AYAR['n_kat']):
    yy = tablo[tablo.fold == kk].year.to_numpy()
    print(f'fold {kk}: {len(yy)} years ({len(yy)/len(yillar):.0%}), {(R_kat == kk).float().mean().item():.1%} of rings; '
          f'blocks starting ' + ', '.join(str(y) for y in yy[::AYAR["blok_yil"]]))

bk = np.full(YIL1 + 2, -1, np.int64); bk[yillar] = blok
R_blok = t(bk, torch.long)[R_yil]

def bolunme(kat):
    """Test: this fold's years (~25%) at all sites. Early stopping: ~10% of the remaining 5-year blocks, drawn at
    random, at all sites (unseen years, as in the test). Train: all other years at all sites.
    No test year is used for training or for early stopping."""
    egit_blok = np.unique(blok[kat_blok[blok] != kat])
    n_ic = max(2, int(round(AYAR['ic_blok_oran'] * len(egit_blok))))
    ic_blok = np.random.default_rng(AYAR['bolme_tohum'] + 100 + kat).choice(egit_blok, n_ic, replace=False)
    test = R_kat == kat; ic = torch.isin(R_blok, t(ic_blok, torch.long))
    egit = (~test & ~ic).nonzero().squeeze(1); dog = (~test & ic).nonzero().squeeze(1); tst = test.nonzero().squeeze(1)
    return _alt(egit, dog, tst)

def bolunme_saha(kat):
    """Unseen-site test: test = all years at the sites of spatial group `kat` (10-degree blocks); early stopping =
    the fixed 10% validation sites outside the test group (also unseen sites); train = all other sites, all years.
    Trees at test sites were never seen, so their tree effect stays at its prior mean (zero)."""
    ts = SITE_KAT[R_site] == kat; ic = IC[R_site] & ~ts
    return _alt((~ts & ~ic).nonzero().squeeze(1), ic.nonzero().squeeze(1), ts.nonzero().squeeze(1))

def _alt(egit, dog, tst):
    if AYAR['hizli_deneme']:
        g = torch.Generator(device='cpu').manual_seed(0)
        sec = lambda x, n: x[torch.randperm(len(x), generator=g)[:n].to(DEV)]
        egit, dog = sec(egit, 200_000), sec(dog, 20_000)
    return egit, dog, tst

In [ ]:
# --- Model: theory-embedded network (equations 1-3 of the manuscript) with options ---
class MLP(nn.Module):
    """Site-parameter network with `katman` hidden layers of `gizli` units (residual blocks beyond the first)."""
    def __init__(s, n_in, gizli, katman, n_out):
        super().__init__()
        s.gir = nn.Linear(n_in, gizli)
        s.bloklar = nn.ModuleList([nn.Sequential(nn.LayerNorm(gizli), nn.Linear(gizli, gizli), nn.GELU()) for _ in range(katman - 1)])
        s.cik = nn.Linear(gizli, n_out)
        with torch.no_grad(): s.cik.weight.mul_(0.1); s.cik.bias.zero_()
    def forward(s, x):
        h = F.gelu(s.gir(x))
        for b in s.bloklar: h = h + b(h)
        return s.cik(h)

class SiteAgi(nn.Module):
    """Genus/species embeddings + site descriptors + site environment (soil, topography, biome, ecoregion, Koppen)."""
    def __init__(s, n_out, gizli, katman):
        super().__init__()
        e = AYAR['emb']
        s.ge = nn.Embedding(NG + 1, e); s.se = nn.Embedding(NS + 1, e)              # last index = unknown taxon
        s.b = nn.Embedding(NB + 1, 4); s.e = nn.Embedding(NE + 1, 4); s.k = nn.Embedding(NK + 1, 4)
        s.mlp = MLP(2 * e + DESC.shape[1] + ENV.shape[1] + 12, gizli, katman, n_out)
    def forward(s, G, gizle=None):
        g, sp, b, e, k = G['g'], G['sp'], G['b'], G['e'], G['k']
        if gizle is not None:                                                   # taxon dropout: learn the 'unknown' embedding
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        if s.training:                                                          # 10% masking of the categorical environment
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        dn = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if s.training else 0)
        en = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if s.training else 0)
        return s.mlp(torch.cat([s.ge(g), s.se(sp), dn, en, s.b(b), s.e(e), s.k(k)], 1))

def monoton(x, a, c, s):
    """Equation (3): non-negative mixture of logistic functions, rising from 0 to 1. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

class TeoriModeli19(nn.Module):
    def __init__(s, vpd=True, opt=False, iklim=True, agac=False, cwd=False, spei=False, gizli=128, katman=2):
        super().__init__()
        J = AYAR['J']; s.J, s.vpd, s.opt, s.iklim, s.agac, s.cwd, s.spei = J, vpd, opt, iklim, agac, cwd, spei
        n = 3 + 6 * J + 24 + (3 * J if vpd else 0) + (2 if opt else 0) + (3 * J if cwd else 0) + (3 * J if spei else 0) + 3 + 1
        s.ag = SiteAgi(n, gizli, katman)
        s.Tbaz = torch.linspace(-5, 25, J, device=DEV); s.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)
        s.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        s.Wbaz = torch.linspace(0.05, 1.5, J, device=DEV); s.Zbaz = torch.linspace(-2.5, 1.5, J, device=DEV)
        s.u = nn.Parameter(torch.zeros(NTREE, device=DEV))                     # tree effect (used only when agac=True)
        s.m_logit = math.log((2 / 3) / (1 - 2 / 3))

    def parametreler(s, G, gizle=None):
        o = s.ag(G, gizle); J = s.J; p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm^2
        p['m'] = torch.sigmoid(o[:, 2] + s.m_logit)                            # starts at 2/3 (metabolic scaling)
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], s.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], s.Mbaz + 0.1 * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if s.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], s.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.cwd:   # climatic water deficit (100 mm/month); response decreases with deficit
            p['aW'], p['cW'], p['sW'] = o[:, i:i+J], s.Wbaz + 0.2 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.spei:  # 12-month standardized water balance; response increases with wetter conditions
            p['aZ'], p['cZ'], p['sZ'] = o[:, i:i+J], s.Zbaz + 0.5 * torch.tanh(o[:, i+J:i+2*J]), 0.2 + 0.5 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.opt:   # temperature optimum: growth declines above a site-specific upper threshold (starts near 25 °C)
            p['cH'], p['sH'] = 25 + 7 * torch.tanh(o[:, i]), 0.5 + 2 * F.softplus(o[:, i+1]); i += 2
        p['sig0'], p['sigC'], p['sigR'] = o[:, i] - 0.3, 0.1 * o[:, i+1], 0.1 * o[:, i+2]; i += 3   # heteroscedastic SD
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1
        return p

    def forward(s, idx, gizle=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = s.parametreler(G, gizle)
        r = p['r0'] + G['rp']; B = math.pi * r ** 2
        x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
        Gp = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20       # equation (1), smoothed
        T, M, D, gEk = X[..., 0], X[..., 2], X[..., 3], X[..., 4]
        if s.iklim:
            gT = monoton(T, p['aT'], p['cT'], p['sT'])
            if s.opt: gT = gT * (1 - torch.sigmoid((T - p['cH'][:, None]) / p['sH'][:, None]))
            gM = monoton(M, p['aM'], p['cM'], p['sM'])
            sinir = torch.minimum(gT, gM)
            gD = 1 - monoton(D, p['aD'], p['cD'], p['sD']) if s.vpd else torch.ones_like(gT)
            sinir = torch.minimum(sinir, gD)                                   # Liebig: min(gT, gM, gD), equation (2)
            if s.cwd: sinir = torch.minimum(sinir, 1 - monoton(X[..., 5], p['aW'], p['cW'], p['sW']))
            if s.spei: sinir = torch.minimum(sinir, monoton(X[..., 6], p['aZ'], p['cZ'], p['sZ']))
        else:
            gT = gM = gD = sinir = torch.ones_like(T)                          # climate-free ablation
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        u = s.u[G['tree']] if s.agac else torch.zeros_like(C)
        BAI = Gp * C * torch.exp(u)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                     # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        ls = p['sig0'] + p['sigC'] * torch.log(C) + p['sigR'] * torch.log(r)  # aleatoric SD varies with climate and size
        sicak = (T > p['cH'][:, None]) if s.opt else torch.zeros_like(T, dtype=torch.bool)   # month above the temperature optimum
        return mu, ls, dict(gT=gT, gM=gM, gD=gD, C=C, G=Gp, w=p['omega'] * gEk, sicak=sicak)

def yap(ad, gizli, katman):
    return TeoriModeli19(vpd=True, opt='_opt' in ad, iklim=not ad.startswith('iklimsiz'), agac=ad.endswith('_re'),
                         cwd='_cwd' in ad, spei='_spei' in ad, gizli=gizli, katman=katman).to(DEV)

def ag_param(m):
    return sum(p.numel() for n, p in m.named_parameters() if n != 'u')

In [ ]:
# --- Training and evaluation ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm']); LOG_ESIK = math.log(0.01)
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

def nll(mu, ls, y):
    """Gaussian NLL of log ring width; zero rings are left-censored below the 0.01 mm resolution."""
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, bs=131072):
    model.eval(); MU, LS = [], []
    for i in range(0, len(idx), bs):
        mu, ls, _ = model(idx[i:i+bs]); MU.append(mu); LS.append(ls)
    model.train(); return torch.cat(MU), torch.cat(LS)

def yillik_sinyal(idx, mu, y):
    """Median site correlation between observed and predicted growth anomalies, averaged per site and year
    (sites with >= 8 test years). 'tum': each series centred on its test-period mean, so slow size/age trends and
    differences between blocks count. 'blok_ici': each series centred within each 5-year test block, which removes
    trends and levels and keeps only year-to-year (high-frequency, mainly climatic) variation."""
    nz = (y > LOG_Y_SIFIR + 1e-6).cpu().numpy()
    df = pd.DataFrame(dict(ser=R_ser[idx].cpu().numpy(), s=R_site[idx].cpu().numpy(), yil=R_yil[idx].cpu().numpy(),
                           b=R_blok[idx].cpu().numpy(), o=y.cpu().numpy(), p=mu.float().cpu().numpy()))[nz]
    sonuc = {}
    for ad, grup in [('tum', ['ser']), ('blok_ici', ['ser', 'b'])]:
        a = df[['s', 'yil']].copy()
        a['o'] = df.o - df.groupby(grup).o.transform('mean'); a['p'] = df.p - df.groupby(grup).p.transform('mean')
        sy = a.groupby(['s', 'yil'])[['o', 'p']].mean().reset_index()
        r = sy.groupby('s').apply(lambda g: g.o.corr(g.p) if len(g) >= 8 else np.nan, include_groups=False).dropna()
        sonuc[ad] = (float(r.median()), int(len(r)))
    return sonuc

In [ ]:
# --- Load the final model: 4 year folds x 5 seeds (Defter 20) ---
import glob, gc, re
f_ = os.path.join(KAYNAK, 'P2_secilen_mimari.json')
GIZLI, KATMAN = (json.load(open(f_))['gizli'], json.load(open(f_))['katman']) if os.path.exists(f_) else (128, 2)

def yukle(kl):
    m = yap(AYAR['model'], GIZLI, KATMAN); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m

UYE = []
for kl in sorted(glob.glob(os.path.join(KAYNAK, f'yil*__{AYAR["model"]}__h{GIZLI}x{KATMAN}__s*'))):
    if os.path.exists(os.path.join(kl, 'bitti.json')) and os.path.exists(os.path.join(kl, 'en_iyi.pt')):
        k_, s_ = re.search(r'yil(\d+)__.*__s(\d+)$', kl).groups(); UYE.append(dict(kat=int(k_), tohum=int(s_), kl=kl))
assert UYE, f'No trained models in {KAYNAK}: run Defter 20 first'
MODELLER = [yukle(u['kl']) for u in UYE]
ALT = [i for i, u in enumerate(UYE) if u['tohum'] == min(v['tohum'] for v in UYE)]   # one member per fold (limiting factors, curves)
print(f'Final model {AYAR["model"]} ({KATMAN} x {GIZLI}): {len(MODELLER)} members; subset for Q4-Q5: {len(ALT)}')

@torch.no_grad()
def ongor(m, idx, bs=131072, ek=False):
    """Predictions with the current global climate tensor OZ. ek=True also returns per-ring limiting-factor shares."""
    m.eval(); MU, EK = [], []
    for i in range(0, len(idx), bs):
        mu, ls, e = m(idx[i:i+bs]); MU.append(mu)
        if ek:
            w = e['w']; ws = w.sum(1).clamp_min(1e-9)
            st = torch.stack([e['gT'], e['gM'], e['gD']], -1).argmin(-1)                # 0 temperature, 1 soil moisture, 2 VPD
            pay = lambda msk: (w * msk).sum(1) / ws
            EK.append(torch.stack([pay((st == 0) & ~e['sicak']), pay((st == 0) & e['sicak']), pay(st == 1), pay(st == 2), e['C']], 1))
    return torch.cat(MU), (torch.cat(EK) if ek else None)

# site-level groups used throughout
site_bolge = np.array(soz['site_bolge'])
gen_ad = np.array(soz['genus'] + ['Unknown'])[np.minimum(d['site_genus'], len(soz['genus']))]
soguk = desc[:, 2] <= np.quantile(desc[:, 2], 0.25); kurak = desc[:, 5] <= np.quantile(desc[:, 5], 0.25)
nemli = desc[:, 5] >= np.quantile(desc[:, 5], 0.75)
GRUP = {'All sites': np.ones(S, bool), 'Coldest 25%': soguk, 'Driest 25%': kurak, 'Wettest 25%': nemli,
        **{f'Region: {b}': site_bolge == b for b in np.unique(site_bolge)}}
cs_ = pd.Series(gen_ad).value_counts()
GRUP_CINS = {f'Genus: {g}': gen_ad == g for g in cs_[cs_ >= 30].index if g != 'Unknown'}
yuzde = lambda x: 100 * (np.exp(x) - 1)
SITE_n = R_site.cpu().numpy(); YIL_n = R_yil.cpu().numpy()
Y_all = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZ_all = Y_all > LOG_Y_SIFIR + 1e-6

In [ ]:
# --- Q0. How much did climate change? (a) CRU: 1990-2019 vs 1901-1960; (b) ISIMIP3a: observed minus counterfactual climate ---
# Counterfactual climate as in Defter 11: the ISIMIP3a obsclim - counterclim difference (Mengel et al. 2021 detrending) is applied
# to the CRU climate the model was trained on; soil moisture is recomputed with the same VS-Lite bucket.
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    Tclim = np.nanmean(T.reshape(T.shape[0], -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for k in range(N):
        Tt, Pt, Lt = T[:, k], P[:, k], L[:, k % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, k] = m
    return M

KL_IS = os.path.join(ANA, 'IKLIM_ISIMIP'); YIL_SON = AYAR['yil_son']
kod2s = {c: i for i, c in enumerate(soz['site_code'])}
def isimip_oku(sen):
    f = os.path.join(KL_IS, f'isimip_{sen}_aylik.csv.gz')
    assert os.path.exists(f), f'Missing {f} (prepared for Defter 11 with ISIMIP_cikar.R)'
    df = pd.read_csv(f, usecols=['studyCode', 'year', 'month', 'tmp', 'pre', 'vpd']); df = df[df.studyCode.isin(list(kod2s))]
    s_ = df.studyCode.map(kod2s).to_numpy(); y_ = df.year.to_numpy() - YIL0; m_ = df.month.to_numpy() - 1
    out = {}
    for v in ['tmp', 'pre', 'vpd']:
        a = np.full((S, NY, 12), np.nan, np.float32); a[s_, y_, m_] = df[v].to_numpy(np.float32); out[v] = a
    return out
OBS = isimip_oku('obsclim'); CFI = isimip_oku('counterclim')
isimip_var = ~np.isnan(OBS['tmp']).all(axis=(1, 2))
dT = np.nan_to_num(OBS['tmp'] - CFI['tmp'])
oranP = np.clip(np.nan_to_num(np.where(OBS['pre'] > 1.0, CFI['pre'] / np.maximum(OBS['pre'], 1e-6), 1.0), nan=1.0), 0.3, 3.0)
dV = np.nan_to_num(OBS['vpd'] - CFI['vpd']); del OBS, CFI; gc.collect()
LATs = d['site_lat'].astype(float)
T_f = np.nan_to_num(clim[..., 0].astype(np.float64)); P_f = np.nan_to_num(clim[..., 1].astype(np.float64)); V_f = np.nan_to_num(clim[..., 3].astype(np.float64))
T_c = T_f - dT; P_c = P_f * oranP; V_c = np.maximum(V_f - dV, 0)
I_tam = isi_indeksi(T_f.reshape(S, -1))
M_f = kova(T_f.reshape(S, -1), P_f.reshape(S, -1), LATs, I_tam); M_c = kova(T_c.reshape(S, -1), P_c.reshape(S, -1), LATs, I_tam)
print('Check: recomputed factual soil moisture vs Defter 1 (max abs diff):', round(float(np.nanmax(np.abs(M_f - soil.reshape(S, -1)))), 4))
OZ_F = OZ
def degistir(kanal):
    Z = OZ_F.clone()
    for c, a in kanal.items():
        a = t(np.asarray(a, np.float32).reshape(S, -1)); Z[..., c] = torch.where(torch.isnan(a), Z[..., c], a)
    return Z
# counterfactual soil moisture as a delta on the factual (Defter 1) soil moisture, so that years without a climate-change
# signal are identical in both scenarios even if the recomputed bucket differs slightly from Defter 1
M_cd = np.clip(soil.reshape(S, -1) + (M_c - M_f), 0.01, 0.76)
OZ_C = degistir({0: T_c, 1: np.log1p(np.maximum(P_c, 0)), 2: M_cd, 3: V_c})      # all channels counterfactual
OZ_T = degistir({0: T_c}); OZ_W = degistir({2: M_cd}); OZ_D = degistir({3: V_c})  # one channel at a time
SENARYO = {'F': OZ_F, 'C': OZ_C, 'T': OZ_T, 'W': OZ_W, 'D': OZ_D}

In [ ]:
# --- Common pieces: rings of 1902-2019, site-year means, the 20 members' saved attribution, 10-degree spatial blocks ---
HALKA = ((R_yil >= YIL0 + 1) & (R_yil <= YIL_SON)).nonzero().squeeze(1)
NYa = YIL_SON - YIL0 + 1
ANAHTAR = R_site[HALKA] * NYa + (R_yil[HALKA] - YIL0)
SAY = torch.bincount(ANAHTAR, minlength=S * NYa).reshape(S, NYa).cpu().numpy(); VAR = SAY > 0
KLA = os.path.join(K21, 'atif')
FK = {k_: [] for k_ in 'CTWD'}
for i in range(len(MODELLER)):
    f = os.path.join(KLA, f'uye{i:02d}.npz')
    if not os.path.exists(f): continue
    z = np.load(f)
    for k_ in 'CTWD':
        if f'fark_{k_}' in z: FK[k_].append(z[f'fark_{k_}'])
FK = {k_: np.stack(v) for k_, v in FK.items() if v}
assert 'C' in FK, f'No attribution files in {KLA}: run Defter 21 first'
print('members with saved attribution:', len(FK['C']))

def donem_ort(A, y0, y1, nmin=3):
    a, b = max(y0 - YIL0, 0), min(y1 - YIL0 + 1, NYa); v = VAR[:, a:b]; n = v.sum(1)
    return np.where(n >= nmin, (A[..., a:b] * v).sum(-1) / np.maximum(n, 1), np.nan)

LAT, LON = d['site_lat'].astype(float), d['site_lon'].astype(float)
BLOK = (np.floor((LAT + 90) / 10) * 100 + np.floor((LON + 180) / 10)).astype(int)
rngB = np.random.default_rng(2033)

def blok_boot(x, blok, fn, B=1000):
    """Spatial block bootstrap: resample 10-degree blocks with replacement; returns 2.5 and 97.5 percentiles of fn"""
    ub = np.unique(blok); idx = {b: np.where(blok == b)[0] for b in ub}; out = []
    for _ in range(B):
        sec = rngB.choice(ub, len(ub)); ii = np.concatenate([idx[b] for b in sec])
        out.append(fn(x[ii]))
    return np.nanpercentile(out, [2.5, 97.5])

def blok_boot2(x, y, blok, fn, B=1000):
    """Paired block bootstrap for a statistic of two aligned site vectors (e.g. ratio of means)"""
    ub = np.unique(blok); idx = {b: np.where(blok == b)[0] for b in ub}; out = []
    for _ in range(B):
        sec = rngB.choice(ub, len(ub)); ii = np.concatenate([idx[b] for b in sec]); out.append(fn(x[ii], y[ii]))
    return np.nanpercentile(out, [2.5, 97.5])

GR_ANA = {g: GRUP[g] for g in ['All sites', 'Coldest 25%', 'Driest 25%', 'Region: Europe', 'Region: North America', 'Region: Asia',
                              'Region: Southern Hemisphere'] if g in GRUP}
E90 = donem_ort(FK['C'], 1990, YIL_SON, nmin=10)          # [members, S], log effect 1990-2019
E90m = E90.mean(0)

In [ ]:
# --- A1. Network and ring-based estimate on the SAME sites, with the SAME summary statistics and the SAME 10-degree blocks ---
import glob as _glob
KAS = AYAR.get('analiz_son') or next(iter(_glob.glob(os.path.join(ANA, '**', 'ANALIZ_SON'), recursive=True) +
                                          _glob.glob('/content/drive/MyDrive/**/ANALIZ_SON', recursive=True)), None)
print('ANALIZ_SON:', KAS)
a1 = []
f_n1a = os.path.join(KAS, 'N1a_uc_olay_kanali_alan.csv') if KAS else None
if f_n1a and os.path.exists(f_n1a):
    H = pd.read_csv(f_n1a)
    print('ring-based site table columns:', list(H.columns))
    kol_id = next(c for c in H.columns if c.lower() in ('studycode', 'study_code', 'site_code', 'site', 'alan', 'kod', 'sitecode'))
    aday_t = [c for c in H.columns if c in ('L_yuzde_yil', 'toplam_yuzde_yil')] or [c for c in H.columns if 'toplam' in c.lower() and 'yuzde' in c.lower()]
    kol_L = aday_t[0]; print(f'using id column {kol_id!r} and site effect column {kol_L!r} (% per year)')
    Lr = np.full(S, np.nan); ix = H[kol_id].astype(str).map({c: i for i, c in enumerate(soz['site_code'])})
    ok_ = ix.notna().to_numpy(); Lr[ix[ok_].astype(int).to_numpy()] = H.loc[ok_, kol_L].to_numpy(float)
    En = yuzde(E90m)                                         # network, % per year
    for kume, km in [('common sites', np.isfinite(En) & np.isfinite(Lr)), ('all network sites', np.isfinite(En)),
                     ('all ring sites', np.isfinite(Lr))]:
        for g, msk in GR_ANA.items():
            m_ = msk & km
            if m_.sum() < 20: continue
            for kaynak, v in [('network', En), ('rings', Lr)]:
                x = v[m_]
                if not np.isfinite(x).all(): continue
                lo_m, hi_m = blok_boot(x, BLOK[m_], np.mean); lo_d, hi_d = blok_boot(x, BLOK[m_], np.median)
                a1.append(dict(site_set=kume, group=g, source=kaynak, n_sites=int(m_.sum()), n_blocks=len(np.unique(BLOK[m_])),
                               mean_pct=float(x.mean()), mean_lo95=float(lo_m), mean_hi95=float(hi_m),
                               median_pct=float(np.median(x)), median_lo95=float(lo_d), median_hi95=float(hi_d)))
            if kume == 'common sites':
                x, y = En[m_], Lr[m_]
                lo, hi = blok_boot2(x, y, BLOK[m_], lambda a, b: a.mean() / b.mean() if abs(b.mean()) > 1e-9 else np.nan)
                rr = np.corrcoef(x, y)[0, 1]; lo_r, hi_r = blok_boot2(x, y, BLOK[m_], lambda a, b: np.corrcoef(a, b)[0, 1])
                a1.append(dict(site_set=kume, group=g, source='network / rings (ratio of means)', n_sites=int(m_.sum()),
                               mean_pct=float(x.mean() / y.mean()), mean_lo95=float(lo), mean_hi95=float(hi)))
                a1.append(dict(site_set=kume, group=g, source='site correlation network vs rings', n_sites=int(m_.sum()),
                               mean_pct=float(rr), mean_lo95=float(lo_r), mean_hi95=float(hi_r)))
    a1 = pd.DataFrame(a1); a1.to_csv(os.path.join(CIKTI, 'A1_ayni_saha.csv'), index=False)
    pd.set_option('display.width', 220)
    print(a1[a1.site_set == 'common sites'].round(3).to_string(index=False))
else:
    print('A1 skipped: N1a_uc_olay_kanali_alan.csv not found')

In [ ]:
# --- A3. Network results with 10-degree spatial-block intervals (all attribution sites, 1990-2019) ---
a3 = []
for g, msk in {**GR_ANA, **{k_: v for k_, v in GRUP_CINS.items() if k_ in ('Genus: Fagus', 'Genus: Cedrus', 'Genus: Pseudotsuga', 'Genus: Larix', 'Genus: Pinus', 'Genus: Picea')}}.items():
    m_ = msk & np.isfinite(E90m)
    if m_.sum() < 20: continue
    x = yuzde(E90m[m_]); lo, hi = blok_boot(x, BLOK[m_], np.median)
    um = yuzde(np.nanmedian(E90[:, m_], axis=1))
    a3.append(dict(group=g, n_sites=int(m_.sum()), n_blocks=len(np.unique(BLOK[m_])), median_pct=float(np.median(x)),
                   block_lo95=float(lo), block_hi95=float(hi), members_p05=float(np.quantile(um, 0.05)), members_p95=float(np.quantile(um, 0.95))))
a3 = pd.DataFrame(a3); a3.to_csv(os.path.join(CIKTI, 'A3_blok_aralik.csv'), index=False)
print(a3.round(2).to_string(index=False))

In [ ]:
# --- A2. Emergence without changes in the site sample: (a) balanced panels, (b) chain-linked consecutive-decade comparisons ---
DEC = list(range(1900, 2020, 10))
ED = np.stack([donem_ort(FK['C'], max(dc, YIL0 + 1), dc + 9, nmin=3) for dc in DEC], -1)     # [members, S, decades]
var_dec = np.isfinite(ED[0])                                                                  # [S, decades]
a2, a2c = [], []
PANEL = {'balanced 1930s-2000s': (1930, 2000), 'balanced 1950s-2010s': (1950, 2010), 'balanced 1910s-1990s': (1910, 1990)}
for pad, (d0, d1) in PANEL.items():
    j = [DEC.index(x) for x in range(d0, d1 + 1, 10)]
    pan = var_dec[:, j].all(1)
    for g, msk in GR_ANA.items():
        m_ = msk & pan
        if m_.sum() < 15: continue
        for jj in j:
            x = ED[:, m_, jj]; med = yuzde(np.median(x.mean(0))); um = yuzde(np.median(x, axis=1))
            lo, hi = blok_boot(yuzde(x.mean(0)), BLOK[m_], np.median, B=500)
            a2.append(dict(method=pad, group=g, decade=DEC[jj], n_sites=int(m_.sum()), median_pct=float(med),
                           members_p05=float(np.quantile(um, 0.05)), members_p95=float(np.quantile(um, 0.95)), block_lo95=float(lo), block_hi95=float(hi)))
# chain-linked index: median change between consecutive decades on the sites present in both, accumulated from the first decade
for g, msk in GR_ANA.items():
    idx_u = None
    for jj, dc in enumerate(DEC):
        if jj == 0:
            m_ = msk & var_dec[:, 0]
            if m_.sum() < 15: break
            idx_u = np.median(ED[:, m_, 0], axis=1); n_ = int(m_.sum())
        else:
            m_ = msk & var_dec[:, jj] & var_dec[:, jj - 1]
            if m_.sum() < 15: break
            idx_u = idx_u + np.median(ED[:, m_, jj] - ED[:, m_, jj - 1], axis=1); n_ = int(m_.sum())
        a2.append(dict(method='chain-linked', group=g, decade=dc, n_sites=n_, median_pct=float(yuzde(np.median(idx_u))),
                       members_p05=float(yuzde(np.quantile(idx_u, 0.05))), members_p95=float(yuzde(np.quantile(idx_u, 0.95)))))
a2 = pd.DataFrame(a2); a2.to_csv(os.path.join(CIKTI, 'A2_denge_onyil.csv'), index=False)
# pre-defined emergence criterion: first decade from which the 5th-95th member range excludes zero in that and every later decade
for (mth, g), x in a2.groupby(['method', 'group']):
    x = x.sort_values('decade'); isaret = -1 if x.median_pct.iloc[-1] < 0 else 1
    disla = (x.members_p95 < 0) if isaret < 0 else (x.members_p05 > 0)
    em = next((int(dc) for k_, dc in enumerate(x.decade) if disla.iloc[k_:].all()), None)
    a2c.append(dict(method=mth, group=g, sign='loss' if isaret < 0 else 'gain', emergence_decade=em,
                    last_decade=int(x.decade.iloc[-1]), last_median_pct=float(x.median_pct.iloc[-1]), n_sites_last=int(x.n_sites.iloc[-1])))
a2c = pd.DataFrame(a2c); a2c.to_csv(os.path.join(CIKTI, 'A2_ortaya_cikis.csv'), index=False)
print(a2c.to_string(index=False))
print(a2[(a2.group == 'Driest 25%')].pivot_table(index='decade', columns='method', values='median_pct').round(2).to_string())

In [ ]:
# --- A4. Is the attribution sensitive to series age and sampling end year? (the century trend of residuals is age/sampling related) ---
ser_n = R_ser.cpu().numpy(); SITE_n = R_site.cpu().numpy(); YIL_n = R_yil.cpu().numpy()
ham = pd.DataFrame(dict(s=d['ring_series'], y=d['ring_year']))
ilk = ham.groupby('s').y.min().reindex(range(NSER)).to_numpy(); son = ham.groupby('s').y.max().reindex(range(NSER)).to_numpy(); del ham
k90 = (YIL_n >= 1990) & (YIL_n <= YIL_SON) & (ilk[ser_n] > YIL0)
yas90 = pd.Series(YIL_n[k90] - ilk[ser_n][k90]).groupby(SITE_n[k90]).median().reindex(range(S)).to_numpy()
son_site = pd.Series(son).groupby(d['series_site']).max().reindex(range(S)).to_numpy()
a4 = []
for deg, v in [('median series age in 1990-2019 (years since first ring)', yas90), ('last ring year of the site', son_site)]:
    for g, msk in {k_: GR_ANA[k_] for k_ in ['All sites', 'Coldest 25%', 'Driest 25%', 'Region: Europe']}.items():
        m_ = msk & np.isfinite(E90m) & np.isfinite(v)
        if m_.sum() < 30: continue
        q = np.quantile(v[m_], [1 / 3, 2 / 3]); sinif = np.searchsorted(q, v, side='right')
        for c_, ad in enumerate(['lower third', 'middle third', 'upper third']):
            mm = m_ & (sinif == c_)
            if mm.sum() < 10: continue
            x = yuzde(E90m[mm]); lo, hi = blok_boot(x, BLOK[mm], np.median, B=500)
            a4.append(dict(variable=deg, group=g, tercile=ad, range=f'{np.min(v[mm]):.0f}-{np.max(v[mm]):.0f}', n_sites=int(mm.sum()),
                           median_pct=float(np.median(x)), block_lo95=float(lo), block_hi95=float(hi)))
        # slope of the site effect on the standardized variable, within the group
        z = (v[m_] - v[m_].mean()) / v[m_].std(); y = yuzde(E90m[m_])
        b = np.polyfit(z, y, 1)[0]; lo, hi = blok_boot2(z, y, BLOK[m_], lambda a, c: np.polyfit(a, c, 1)[0], B=500)
        a4.append(dict(variable=deg, group=g, tercile='slope per s.d. (% per year)', n_sites=int(m_.sum()), median_pct=float(b),
                       block_lo95=float(lo), block_hi95=float(hi)))
a4 = pd.DataFrame(a4); a4.to_csv(os.path.join(CIKTI, 'A4_yas_orneklem.csv'), index=False)
print(a4.round(2).to_string(index=False))

In [ ]:
# --- A5. Bounds in the counterfactual climate: how often were they active, and does relaxing them change the result? ---
OBS = isimip_oku('obsclim'); CFI = isimip_oku('counterclim')
p90 = slice(1990 - YIL0, YIL_SON - YIL0 + 1)
gecerli = np.isfinite(OBS['pre']) & np.isfinite(CFI['pre'])
ham_oran = np.where(OBS['pre'] > 1.0, CFI['pre'] / np.maximum(OBS['pre'], 1e-6), np.nan)
say = dict(
    months_with_isimip=int(gecerli.sum()),
    precip_ratio_set_to_1_obs_below_1mm=int((gecerli & (OBS['pre'] <= 1.0)).sum()),
    precip_ratio_below_0_3=int((ham_oran < 0.3).sum()), precip_ratio_above_3=int((ham_oran > 3.0).sum()),
    vpd_counterfactual_floored_at_0=int(((V_f - dV) < 0).sum()),
    soil_counterfactual_at_lower_bound=int(((soil.reshape(S, -1) + (M_c - M_f)) < 0.01).sum()),
    soil_counterfactual_at_upper_bound=int(((soil.reshape(S, -1) + (M_c - M_f)) > 0.76).sum()))
say_90 = {f'{k_} (1990-2019)': int(v[:, p90].sum()) for k_, v in [
    ('precip_ratio_set_to_1_obs_below_1mm', gecerli & (OBS['pre'] <= 1.0)), ('precip_ratio_below_0_3', ham_oran < 0.3),
    ('precip_ratio_above_3', ham_oran > 3.0)]}
a5s = pd.DataFrame([dict(item=k_, months=v) for k_, v in {**say, **say_90}.items()])
a5s['share_of_isimip_months'] = a5s.months / max(say['months_with_isimip'], 1)
a5s.to_csv(os.path.join(CIKTI, 'A5_sinirlama_sayim.csv'), index=False); print(a5s.to_string(index=False))
# relaxed counterfactual: precipitation ratio bounded to 0.1-10 instead of 0.3-3 (soil moisture stays within the bucket's physical range)
oranP_gevsek = np.clip(np.nan_to_num(np.where(OBS['pre'] > 1.0, CFI['pre'] / np.maximum(OBS['pre'], 1e-6), 1.0), nan=1.0), 0.1, 10.0)
del OBS, CFI, ham_oran; gc.collect()
P_cg = P_f * oranP_gevsek
M_cg = kova(T_c.reshape(S, -1), P_cg.reshape(S, -1), LATs, I_tam)
M_cdg = np.clip(soil.reshape(S, -1) + (M_cg - M_f), 0.01, 0.76)
OZ_CG = degistir({0: T_c, 1: np.log1p(np.maximum(P_cg, 0)), 2: M_cdg, 3: V_c})
H90 = ((R_yil >= 1990) & (R_yil <= YIL_SON)).nonzero().squeeze(1)
K90 = R_site[H90]; N90 = torch.bincount(K90, minlength=S).float()
def alan_ort(v):
    return (torch.bincount(K90, weights=v.float(), minlength=S) / N90.clamp_min(1)).cpu().numpy()
SAY90 = torch.bincount(K90 * NYa + (R_yil[H90] - YIL0), minlength=S * NYa).reshape(S, NYa)[:, 1990 - YIL0:].gt(0).sum(1).cpu().numpy()
UYE4 = ALT                                                   # one member per fold
def ongor_sen(Z, i):
    global OZ
    OZ = Z; mu = ongor(MODELLER[i], H90)[0]; OZ = OZ_F; return mu
a5 = []; ana_cf, gev_cf = [], []
for i in UYE4:
    muF = ongor_sen(OZ_F, i)
    ana_cf.append(alan_ort(muF - ongor_sen(OZ_C, i))); gev_cf.append(alan_ort(muF - ongor_sen(OZ_CG, i)))
    torch.cuda.empty_cache()
ana_cf, gev_cf = np.mean(ana_cf, 0), np.mean(gev_cf, 0); ok5 = SAY90 >= 10
for g, msk in GR_ANA.items():
    m_ = msk & ok5
    if m_.sum() < 20: continue
    a5.append(dict(group=g, n_sites=int(m_.sum()), median_pct_main=float(yuzde(np.median(ana_cf[m_]))),
                   median_pct_relaxed_precip_bounds=float(yuzde(np.median(gev_cf[m_]))),
                   difference_pct_points=float(yuzde(np.median(gev_cf[m_])) - yuzde(np.median(ana_cf[m_])))))
a5 = pd.DataFrame(a5); a5.to_csv(os.path.join(CIKTI, 'A5_sinirlama_duyarlilik.csv'), index=False); print(a5.round(2).to_string(index=False))
del OZ_CG; torch.cuda.empty_cache()

In [ ]:
# --- A6. Full factorial decomposition: temperature (T), moisture (W: precipitation + soil moisture) and VPD (D), 2^3 = 8 climates.
#     v(S) = prediction with the factors in S at their observed and the others at their counterfactual values, minus the
#     all-counterfactual prediction; Shapley value of each factor = weighted average of its marginal contributions. ---
from itertools import combinations
FAKT = {'T': {0: T_c}, 'W': {1: np.log1p(np.maximum(P_c, 0)), 2: M_cd}, 'D': {3: V_c}}
def iklim(gozlenen):
    """Climate tensor with the factors in `gozlenen` observed and the others counterfactual"""
    deg = {}
    for f_, kanallar in FAKT.items():
        if f_ not in gozlenen: deg.update(kanallar)
    return degistir(deg) if deg else OZ_F
alt_kume = [frozenset(c) for r in range(4) for c in combinations('TWD', r)]
V = {s_: [] for s_ in alt_kume}
for i in UYE4:
    mu = {}
    for s_ in alt_kume:
        Z = iklim(s_); mu[s_] = ongor_sen(Z, i); del Z
    for s_ in alt_kume: V[s_].append(alan_ort(mu[s_] - mu[frozenset()]))
    del mu; torch.cuda.empty_cache()
V = {s_: np.mean(v, 0) for s_, v in V.items()}               # [S] site effects (log), mean over members
import math as _m
def shapley(f_):
    tot = 0
    for s_ in alt_kume:
        if f_ in s_: continue
        w = _m.factorial(len(s_)) * _m.factorial(3 - len(s_) - 1) / _m.factorial(3)
        tot = tot + w * (V[s_ | {f_}] - V[s_])
    return tot
SH = {f_: shapley(f_) for f_ in 'TWD'}; TOP = V[frozenset('TWD')]
# Defter 21 definition for reference: one factor counterfactual at a time, starting from the observed climate
TEK = {f_: TOP - V[frozenset('TWD') - {f_}] for f_ in 'TWD'}
a6, AD = [], {'T': 'temperature', 'W': 'moisture (precipitation + soil moisture)', 'D': 'VPD'}
for g, msk in {**GR_ANA, **{k_: v for k_, v in GRUP_CINS.items() if k_ in ('Genus: Fagus', 'Genus: Cedrus', 'Genus: Pseudotsuga', 'Genus: Larix')}}.items():
    m_ = msk & ok5
    if m_.sum() < 15: continue
    sat = dict(group=g, n_sites=int(m_.sum()), total_mean_pct=float(yuzde(TOP[m_].mean())), total_median_pct=float(yuzde(np.median(TOP[m_]))))
    for f_ in 'TWD':
        sat[f'shapley_{AD[f_]}_mean_pct'] = float(yuzde(SH[f_][m_].mean()))
        sat[f'only_{AD[f_]}_observed_mean_pct'] = float(yuzde(V[frozenset(f_)][m_].mean()))
        sat[f'only_{AD[f_]}_counterfactual_mean_pct'] = float(yuzde(TEK[f_][m_].mean()))
    sat['interaction_vs_only_observed_mean_pct'] = float(yuzde(TOP[m_].mean()) - sum(yuzde(V[frozenset(f_)][m_].mean()) for f_ in 'TWD'))
    a6.append(sat)
a6 = pd.DataFrame(a6); a6.to_csv(os.path.join(CIKTI, 'A6_faktoriyel.csv'), index=False)
pd.DataFrame(dict(studyCode=soz['site_code'], n_years_1990_2019=SAY90, total=yuzde(TOP),
                  **{f'shapley_{f_}': yuzde(SH[f_]) for f_ in 'TWD'}, **{'v_' + ''.join(sorted(s_)) if s_ else 'v_none': yuzde(V[s_]) for s_ in alt_kume})
             ).to_csv(os.path.join(CIKTI, 'A6_faktoriyel_alan.csv'), index=False)
print(a6.round(2).T.to_string())

In [ ]:
# --- A7. How far do the counterfactual inputs lie outside the observed climate of each site? Share of growing-season months
#     of 1990-2019 in which counterfactual T, soil moisture or VPD falls outside the site's observed 1902-2024 range for that
#     calendar month (and outside the global training range). ---
gw = np.broadcast_to(gE[:, None, :], (S, NY, 12))
Tf3, Mf3_, Vf3 = T_f.reshape(S, NY, 12), soil.reshape(S, NY, 12), V_f.reshape(S, NY, 12)
Tc3, Mc3_, Vc3 = T_c.reshape(S, NY, 12), M_cd.reshape(S, NY, 12), V_c.reshape(S, NY, 12)
a7 = []
for ad, F3, C3 in [('temperature', Tf3, Tc3), ('soil moisture', Mf3_, Mc3_), ('VPD', Vf3, Vc3)]:
    lo, hi = np.nanmin(F3[:, 1:], axis=1, keepdims=True), np.nanmax(F3[:, 1:], axis=1, keepdims=True)
    disari = ((C3 < lo) | (C3 > hi))[:, p90]; agirlik = (gw[:, p90] > 0.1)
    pay = (disari & agirlik).sum((1, 2)) / np.maximum(agirlik.sum((1, 2)), 1)
    glo, ghi = np.nanmin(F3), np.nanmax(F3); gdis = float(((C3[:, p90] < glo) | (C3[:, p90] > ghi)).mean())
    for g, msk in GR_ANA.items():
        m_ = msk & ok5
        if m_.sum() < 20: continue
        a7.append(dict(variable=ad, group=g, n_sites=int(m_.sum()), median_share_outside_site_range=float(np.median(pay[m_])),
                       p90_share_outside_site_range=float(np.quantile(pay[m_], 0.9)), share_outside_global_range=gdis))
a7 = pd.DataFrame(a7); a7.to_csv(os.path.join(CIKTI, 'A7_dagilim_disi.csv'), index=False); print(a7.round(4).to_string(index=False))

In [ ]:
print(sorted(f for f in os.listdir(CIKTI) if f.endswith('.csv')))
print('==== DEFTER 23A TAMAM ====')